# Stage 4 — Module-based workflow (git pull + reload, not notebook re-open)

Uses the extracted candidate modules in `src/stage4_symbol_detection/` instead of
inline notebook code. After any fix is pushed to the repo, just re-run the **Update**
cell below (`git pull` + reload) instead of re-opening this notebook from GitHub.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo (first time only — see Update cell below for later runs)

In [ ]:
!git clone https://github.com/TomGeorge45/pid-opensrc-substitution.git /content/pid-ml

## 3. Install dependencies

Molmo2-O-7B requires `transformers==4.57.1` specifically (see
`src/stage4_symbol_detection/molmo_candidate.py` docstring). Qwen3-VL and InternVL3 were
verified working on a newer transformers (5.12.1) — if you only need those two, you can
skip this exact pin and install a recent `transformers` instead. This cell installs the
Molmo-compatible version since that's what's being tested right now.

In [ ]:
!pip install -q transformers==4.57.1 accelerate torch pycocotools supervision \
    kagglehub kaggle qwen-vl-utils einops timm

## ⚠️ Restart the runtime now

Runtime → Restart session. Then continue from the cell below — no need to re-run 1-3
unless you also need to re-clone/re-install (Drive stays mounted and the repo stays on
disk across a restart within the same runtime).

## 4. Path setup + imports

In [ ]:
import sys
sys.path.insert(0, '/content/pid-ml/src')

from stage4_symbol_detection import data_utils, molmo_candidate

## 5. Load Molmo2-O-7B

In [ ]:
processor, model = molmo_candidate.load()

## 6. Round-trip test on a sample tile

In [ ]:
img, img_path, label_path, gt_count = data_utils.load_sample()
print(f"Sample: {img_path.name} | ground truth boxes: {gt_count}")

raw_text, latency = molmo_candidate.run(processor, model, img)
print(f"Latency: {latency:.2f}s")
print(f"\nRaw output:\n{raw_text[:1500]}")

detections, error = molmo_candidate.parse(raw_text, img.width, img.height)
if error:
    print(f"\n❌ PARSE FAILED: {error}")
else:
    print(f"\n✓ Parsed {len(detections)} detections")
    for d in detections[:8]:
        print(" ", d)

## 7. Dev-set parse-rate check (checklist 2.5)

In [ ]:
from stage4_symbol_detection import eval_harness

dev_set = eval_harness.build_dev_set()
results, parse_rate = eval_harness.run_parse_check(
    dev_set,
    run_fn=lambda img: molmo_candidate.run(processor, model, img),
    parse_fn=molmo_candidate.parse,
    needs_dims=True,
)

---
## Update cell — run this after any future fix is pushed, instead of re-opening from GitHub

`git pull` fetches the latest code onto disk; `importlib.reload` picks it up in this
already-running process without needing a full restart (unless the fix touches something
only read at import time, e.g. a new top-level dependency — restart in that case).

In [ ]:
!cd /content/pid-ml && git pull

import importlib
from stage4_symbol_detection import data_utils, molmo_candidate, eval_harness
importlib.reload(data_utils)
importlib.reload(molmo_candidate)
importlib.reload(eval_harness)
print("Reloaded.")